## MCP 서버에서 도구 노출하기

다른 에이전트가 도구나 데이터에 접근할 수 있도록 하려면 MCP 서버를 활용할 수 있습니다.

Databricks는 3가지 유형의 MCP를 지원합니다:
	
- **Databricks 관리형 MCP 서버**:
Databricks는 에이전트가 Unity Catalog의 데이터와 도구를 쿼리할 수 있는 즉시 사용 가능한 서버를 제공합니다. 이를 통해 Databricks 서비스를 MCP 리소스로 노출합니다(Databricks Vector Search, UC 함수, Genie 스페이스, DBSQL 등).

- **외부 MCP 서버**:
관리형 프록시와 Unity Catalog 연결을 사용하여 Databricks 외부에 호스팅된 타사 MCP 서버에 연결합니다.

- **커스텀 MCP 서버**:
자체 서버를 가져오거나 타사 MCP 서버를 실행하기 위해 Databricks 앱으로 자체 MCP 서버를 안전하게 호스팅합니다.





자세한 내용은 Databricks [관리형 MCP 서버 문서](https://docs.databricks.com/aws/en/generative-ai/mcp/managed-mcp#available-managed-servers)를 참조하세요.


이 예제에서는 UC 시스템 ai 함수와 같은 Databricks 관리형 MCP 서버의 도구를 에이전트에 제공하는 방법을 보여드립니다. 그런 다음 Databricks 관리형 MCP 서버 또는 커스텀 서버 등 다른 MCP 도구로 에이전트를 구성할 수 있습니다!

In [0]:
%pip install -U -qqqq mlflow>=3.1.4 langchain==0.3.27 langgraph==0.6.11 databricks-langchain pydantic databricks-agents unitycatalog-langchain[databricks] databricks-feature-engineering==0.12.1 protobuf<5  cryptography<43 databricks-mcp
dbutils.library.restartPython()

In [0]:
%run ../_resources/01-setup

## Databricks 관리형 MCP

Databricks 관리형 MCP 서버에서 UC 시스템 ai 함수를 사용하려면 [agent_config.yaml](https://adb-984752964297111.11.azuredatabricks.net/editor/files/2254899697811646?o=984752964297111) 파일에 관련 URL을 추가하기만 하면 됩니다.

- **Unity Catalog 시스템 ai 함수**: https://{workspace-hostname}/api/2.0/mcp/functions/system/ai

In [0]:
from databricks.sdk import WorkspaceClient
import os, sys, yaml, mlflow
import nest_asyncio
nest_asyncio.apply()

# --- 경로 설정 ---
agent_eval_path = os.path.abspath(os.path.join(os.getcwd(), "../02-agent-eval"))
sys.path.append(agent_eval_path)
conf_path = os.path.join(agent_eval_path, "agent_config.yaml")

# --- Databricks SDK를 사용하여 워크스페이스 URL 탐지 ---
ws = WorkspaceClient()
workspace_url = ws.config.host.rstrip("/") 
mcp_url = f"{workspace_url}/api/2.0/mcp/functions/system/ai"

# ==========================
# MCP를 위한 설정 업데이트
# ==========================
try:
    config = yaml.safe_load(open(conf_path))
    config["config_version_name"] = "model_with_mcp"
    config["mcp_server_urls"] = [mcp_url]

    with open(conf_path, "w") as f:
        yaml.safe_dump(config, f, sort_keys=False)

except Exception as e:
    print(f"MCP 업데이트 건너뜀: {e}")

## 에이전트 인스턴스 생성

다음으로 에이전트를 인스턴스화하고, 관리형 MCP 서버의 도구가 제공되었는지 확인하기 위해 에이전트가 사용할 수 있는 모든 도구를 확인합니다. 아래 목록에서 생성한 UC 함수 외에도 MCP 서버의 도구(UC 시스템 ai 함수, 즉 **system__ai__python_exec**)를 볼 수 있습니다.

In [0]:
# ==========================
# 에이전트 인스턴스 생성
# ==========================
from agent import LangGraphResponsesAgent
import mlflow.models

model_config = mlflow.models.ModelConfig(development_config=conf_path)

AGENT = LangGraphResponsesAgent(
    uc_tool_names=model_config.get("uc_tool_names"),
    llm_endpoint_name=model_config.get("llm_endpoint_name"),
    system_prompt=model_config.get("system_prompt"),
    mcp_server_urls=model_config.get("mcp_server_urls"),
    max_history_messages=model_config.get("max_history_messages"),
)

print("✅ 사용 가능한 도구:", AGENT.list_tools())

## AI Playground에서 에이전트 테스트

에이전트를 테스트하려면 원하는 엔드포인트를 선택하고 필요한 도구를 추가하면 됩니다. 우리의 경우 관리형 MCP 서버 옵션에서 UC 함수 툴박스의 시스템 ai 함수를 선택하여 도구를 추가합니다. 마찬가지로 커스텀 또는 외부 MCP 서버가 있는 경우 원하는 도구를 추가할 수 있습니다.

<img src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/product/ai-agent/pg-mcp-img1.png" width="800px">

질문을 시작하면 에이전트가 위에서 추가한 관리형 MCP 서버의 도구를 제대로 로드했음을 알 수 있습니다.

<img src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/product/ai-agent/pg-mcp-img2.png" width="800">

이제 더 자세히 탐색해 볼 수 있습니다...


<img src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/product/ai-agent/pg-mcp-img3.png" width="800">

## 다음 단계
Databricks의 다른 MCP 서버 옵션을 더 탐색하고 싶다면 [Databricks MCP 문서](https://docs.databricks.com/aws/en/generative-ai/mcp/)를 참조하세요.